## Semantic Chunking

In [ ]:
pip freeze

In [ ]:
import fitz
from langchain_core.documents import Document
from langchain_openai import AzureOpenAIEmbeddings
from langchain_openai import AzureChatOpenAI
from langchain_experimental.text_splitter import SemanticChunker
from langchain_chroma import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
import os
import pandas as pd
import re
import os
from dotenv import load_dotenv
load_dotenv()  # reads .env so os.environ has your keys


##### Fitz Functions

In [ ]:
name="Texas Family Code"

In [ ]:
pdf=fitz.open(f'./pdfs/{name}.pdf')
#Get Metadata
pdf.metadata
#Get Table of Content
pdf.get_toc()
#Get Page Count
pdf.page_count
#Get Page
pdf.load_page(0).get_text()

In [ ]:
#Create Empty Data Frame
df = pd.DataFrame(columns=['Text', 'Page', 'Vertical Pos', 'Horizontal Pos', 'Font', 'Font Size'])

In [ ]:
#Populate Data Frame
for i in range(0,pdf.page_count):
    for block in pdf[i].get_text("dict")["blocks"]:
        #print(block['lines'])
        for lines in block['lines']:
            #print(lines['spans'])
            for span in lines['spans']:
                data={'Text':[span['text']], 'Page':[i], 'Vertical Pos':[span['origin'][1]], 'Horizontal Pos':[span['origin'][0]], 'Font':[span['font']], 'Font Size':[span['size']]}
                df_new_rows = pd.DataFrame(data)
                df = pd.concat([df, df_new_rows])

In [ ]:
#Sort DF
df_sorted=df.sort_values(['Page','Vertical Pos','Horizontal Pos'])

In [ ]:
#Remove Header & Footer
df_removed=df_sorted[df_sorted['Vertical Pos'] != 21.0]
df_removed=df_removed[df_removed['Vertical Pos'] != 744.0]

In [ ]:
#Create Index
df_index=df_removed.reset_index()
df_index.drop(columns='index',axis=1,inplace=True)

In [ ]:
#Remove Family Code 1st Row
df_index = df_index.iloc[1:].reset_index(drop=True)

In [ ]:
# Sort again for safety
df_index = df_index.sort_values(by=['Page', 'Vertical Pos', 'Horizontal Pos']).reset_index(drop=True)

In [ ]:
#Create Empty Variables
documents = []
pending_metadata = None
pending_text = []
hierarchy_levels = ['title', 'subtitle', 'chapter', 'subchapter', 'part', 'section']
current_context = dict.fromkeys(hierarchy_levels)

In [ ]:
data=df_index.copy()

In [ ]:
def get_hierarchy(text, font, hori, idx, df):
    # Bold levels
    if font == 'Courier-Bold':
        check_text=df.iloc[idx+1]["Text"]
        complete_header=True
        print("Bold")
        if re.match(r'[A-Z0-9\'\-\s;:,()&/]+$',check_text) and not re.match(r'^SUBTITLE\s+[A-Z]\. [A-Z0-9\'\-\s;:,()&/]+$', check_text) and not re.match(r'^CHAPTER\s+\d+[A-Z]?\. [A-Z0-9\'\-\s;:,()&/]+$', check_text) and not re.match(r'^SUBCHAPTER\s+[A-Z](?:-\d+)?\. [A-Z0-9\'\-\s;:,()&/]+$', check_text) and not re.match(r'^PART\s+\d+(-?[A-Z])?\.\s+[A-Z0-9\'\-\s;:,()&/]+$', check_text) and not re.match(r'^\d+$', check_text) and check_text.strip()!="":
            complete_header=False
            print("incomplete")
            if re.match(r'^TITLE\s+\d+(-?[A-Z])?\.\s+[A-Z0-9\'\-\s;:,()&/]+$', text):
                part_header=re.match(r'[A-Z0-9\'\-\s;:,()&/]+$',check_text)
                cat=text.split('.', 1)[1].strip()+" "+part_header[0].strip()
                complete_header=True
                return 'title', cat, None, idx+1
            elif re.match(r'^SUBTITLE\s+[A-Z]\. [A-Z0-9\'\-\s;:,()&/]+$', text):
                part_header=re.match(r'[A-Z0-9\'\-\s;:,()&/]+$',check_text)
                cat=text.split('.', 1)[1].strip()+" "+part_header[0].strip()
                complete_header=True
                return 'subtitle', cat, None, idx+1
            elif re.match(r'^CHAPTER\s+\d+[A-Z]?\. [A-Z0-9\'\-\s;:,()&/]+$', text):
                part_header=re.match(r'[A-Z0-9\'\-\s;:,()&/]+$',check_text)
                cat=text.split('.', 1)[1].strip()+" "+part_header[0].strip()
                complete_header=True
                return 'chapter', cat, None, idx+1
            elif re.match(r'^SUBCHAPTER\s+[A-Z](?:-\d+)?\. [A-Z0-9\'\-\s;:,()&/]+$', text):
                part_header=re.match(r'[A-Z0-9\'\-\s;:,()&/]+$',check_text)
                cat=text.split('.', 1)[1].strip()+" "+part_header[0].strip()
                complete_header=True
                return 'subchapter', cat, None, idx+1
            elif re.match(r'^PART\s+\d+(-?[A-Z])?\.\s+[A-Z0-9\'\-\s;:,()&/]+$', text):
                part_header=re.match(r'[A-Z0-9\'\-\s;:,()&/]+$',check_text)
                cat=text.split('.', 1)[1].strip()+" "+part_header[0].strip()
                complete_header=True
                return 'part', cat, None, idx+1

        elif re.match(r'^TITLE\s+\d+(-?[A-Z])?\.\s+[A-Z0-9\'\-\s;:,()&/]+$', text):
            print("title")
            cat = text.split('.', 1)[1].strip()
            return 'title', cat, None, idx
                
        elif re.match(r'^SUBTITLE\s+[A-Z]\. [A-Z0-9\'\-\s;:,()&/]+$', text):
            print("subtitle")
            cat = text.split('.', 1)[1].strip()
            return 'subtitle', cat, None, idx
                
        elif re.match(r'^CHAPTER\s+\d+[A-Z]?\. [A-Z0-9\'\-\s;:,()&/]+$', text):
            print("chapter")
            cat = text.split('.', 1)[1].strip()
            return 'chapter', cat, None, idx
                
        elif re.match(r'^SUBCHAPTER\s+[A-Z](?:-\d+)?\. [A-Z0-9\'\-\s;:,()&/]+$', text):
            print("subchapter")
            cat = text.split('.', 1)[1].strip()
            return 'subchapter', cat, None, idx
        elif re.match(r'^PART\s+\d+(-?[A-Z])?\.\s+[A-Z0-9\'\-\s;:,()&/]+$', text):
            print("part")
            cat = text.split('.', 1)[1].strip()
            return 'part', cat, None, idx
        elif text.strip()=="":
            print("empty")
            return None,None,text,idx
        elif (re.match(r'^(?=.*[a-z])[A-Za-z0-9\'\-\s;:.,()&/–—§]+$', text) or re.match(r'^\d+$', text)):
            print("correct")
            return None,None,text,idx
            
            
    # Section
    elif font == 'Courier' and 85<=hori <= 95:
        full_section = re.match(r'^(Sec\.\s+\d+\.\d+\.\s+[A-Z0-9\'\-\s;:,&()/]+?\.)\s*(.*)', text)
        part_section=  re.match(r'^(Sec\.\s+\d+\.\d+\.\s+[A-Z0-9\'\-\s;:,()&/]+\.?)', text)
        if full_section:
            section_title = full_section.group(1).strip()
            remaining_text = full_section.group(2).strip()
            return 'section', section_title, remaining_text, idx
        elif part_section:
            section_title=part_section.group(1).strip()
            complete=False
            idx=idx+1
            while complete==False:
                middle=re.match(r'^([A-Z0-9\s\'\-\;\:\,\/&()]+\.?)\s*(.*)', df.iloc[idx]["Text"])
                final=re.match(r'^([A-Z0-9\s\'\-\;\:\,\/&()]+\.?)\s*(.*)', df.iloc[idx]["Text"])
                if final:
                    section_title = section_title+" "+final.group(1).strip()
                    remaining_text=final.group(2).strip()
                    complete=True
                    return 'section', section_title, remaining_text, idx
                elif middle:
                    section_title = section_title+" "+middle.group(1).strip()
        else:
            return None,None,text,idx
    else:
        return None,None,text,idx


In [ ]:
idx=0
#row_count=10
row_count=len(data)
pending_text = []
hierarchy=""
cat=""
remaining_text=""
documents=[]
while idx<row_count:
    text = data.iloc[idx]['Text']
    font = data.iloc[idx]['Font']
    hori = data.iloc[idx]['Horizontal Pos']
    page = data.iloc[idx]['Page']
    print(idx)
    hierarchy, cat, remaining_text,idx = get_hierarchy(text, font, hori, idx, data)
    idx+=1
    if hierarchy == 'section':
        # Finalize previous section
        if pending_metadata:
            documents.append(Document(
                page_content="\n".join(pending_text),
                metadata=pending_metadata
            ))
            pending_text = []

        # Update context
        current_context[hierarchy] = cat
        lower_index = hierarchy_levels.index(hierarchy) + 1
        for level in hierarchy_levels[lower_index:]:
            current_context[level] = None

        # Start new section
        pending_metadata = {
            "source": "Texas Family Code",
            "page": page,
            **{k: v for k, v in current_context.items() if v}
        }

        if remaining_text:
            pending_text.append(remaining_text)
    elif hierarchy:
        current_context[hierarchy] = cat
        lower_index = hierarchy_levels.index(hierarchy) + 1
        for level in hierarchy_levels[lower_index:]:
            current_context[level] = None
    elif remaining_text:
        pending_text.append(remaining_text)
    else:
        if text:
            #print(text)
            pending_match=re.match(r'^([A-Z0-9\s\'\-\;\:\,\/&()]+\.)\s*(.*)', text)
            if pending_match:
                text = pending_match.group(2).strip()
        pending_text.append(text)


# Finalize last section
if pending_metadata:
    documents.append(Document(
        page_content="\n".join(pending_text),
        metadata=pending_metadata
    ))

In [ ]:
documents[-1]

In [ ]:
# Initialize & Run Semantic Chunker to Break Text into Semantic Chunks
text_splitter = SemanticChunker(AzureOpenAIEmbeddings(
        openai_api_type="azure",
        openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        deployment="model3large",
        model="text-embedding-3-large",
        chunk_size=1),
        breakpoint_threshold_type="gradient")

In [ ]:
docs=text_splitter.split_documents(documents)
print(f"{name}: Text Split")

In [ ]:
docs

In [ ]:
# Initialize & Run Embeddings Model to Create Embeddings and Store on Harddrive
embeddings = AzureOpenAIEmbeddings(
        openai_api_type="azure",
        openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        deployment="model3large",
        model="text-embedding-3-large",
        chunk_size=1)  
knowledge_base3 = Chroma.from_documents(documents=docs, persist_directory=f"{name}.pdf",embedding=embeddings)

In [ ]:
# Airtable token is no longer stored here.
# Set AIRTABLE_API_TOKEN in your .env file instead.


In [ ]:
retr=knowledge_base3.as_retriever()
context=retr.invoke("How do i get married?")

In [ ]:
context

In [ ]:
def get_hierarchy(text, font,hori):
    text = text
    
    # Bold levels
    if font == 'Courier-Bold':
        if re.match(r'^TITLE\s+\d+(-[A-Z])?\.\s+[A-Z\s]+$', text):
            cat=text.split('.', 1)[1].strip() 
            return 'title',cat
        elif re.match(r'^SUBTITLE\s+[A-Z]\. [A-Z\s]+$', text):
            cat=text.split('.', 1)[1].strip() 
            return 'subtitle',cat
        elif re.match(r'^CHAPTER\s+\d+\. [A-Z\s]+$', text):
            cat=text.split('.', 1)[1].strip() 
            return 'chapter',cat
        elif re.match(r'^SUBCHAPTER\s+[A-Z]\. [A-Z\s]+$', text):
            cat=text.split('.', 1)[1].strip() 
            return 'subchapter',cat
    
    # Section (plain Courier)
    elif font == 'Courier' and hori == 90:
        if re.match(r'^Sec\.\s+\d+\.\d+\.\s+[A-Z\s]+\.', text):
          
            match = re.match(r'^Sec\.\s+\d+\.\d+\.\s+[A-Z\s]+\.', text)
            matched_part = match.group().split('.',3)[3].strip()
            #print("Matched:", matched_part)
            #unmatched_part = text[len(matched_part):].lstrip()  # Remove leading spaces
            #print("Matched:", matched_part)
            #print("Unmatched:", unmatched_part)
            return 'section',matched_part

    return None,None

In [ ]:
# # Create Document with Metadata
document=[]
   
hierarchy_levels = ['title', 'subtitle', 'chapter', 'subchapter', 'section']
current_context = dict.fromkeys(hierarchy_levels)
documents = []
current_text = []
current_page = 0

for idx, row in temp.iterrows():
     text = row['Text']
     font = row['Font']
     hori = row['Horizontal Pos']
     page = row['Page']
    
     hierarchy,cat = get_hierarchy(text, font, hori)
     print (f"hierarchy:{hierarchy}")
     print (f"cat:{cat}")
     if hierarchy:
        if current_text:
            print(f"current text:{current_text}")
            documents.append(Document(
                page_content="\n".join(current_text),
                metadata={
                    "source": "Texas Family Code",
                    "page": current_page,
                    **{k: v for k, v in current_context.items() if v}
                }
            ))
            current_text = []
            

        # Update current level and reset lower levels
        #if hierarchy==None:
        #    current_text.append(text)
        #    current_page = page
         
        current_context[hierarchy] = cat
        lower_index = hierarchy_levels.index(hierarchy) + 1
        for level in hierarchy_levels[lower_index:]:
            current_context[level] = None
             
     else:
        current_text.append(text)
        current_page = page

# Final block
if current_text:
    documents.append(Document(
        page_content="\n".join(current_text),
        metadata={
            "source": "Texas Family Code",
            "page": current_page,
            **{k: v for k, v in current_context.items() if v}
        }
    ))

In [ ]:
current_context

In [ ]:
documents

In [ ]:
pattern=r'^Sec\.\s+\d+\.\d+\.\s+[A-Z\s]+\.'
string='Sec. 1.002.  COURT.  "Court" means the district court, juvenile'

In [ ]:
not_matched, matched = string[:match.start()], match.group()


In [ ]:
temm= 'Sec. 1.003. SUIT FOR DISSOLUTION OF MARRIAGE. "Suit for dissolution of a marriage" includes a suit for divorce or annulment or to declare a marriage void.'

In [ ]:
matches = re.findall(r'Sec\.\s*\d+\.\d+\.\s+[A-Z .]+\.\s*(.*)', temm)

In [ ]:
matches

In [ ]:
pattern = r'^Sec\.\s+\d+\.\d+\.\s+[A-Z\s]+\.'
string = 'Sec. 1.002.  COURT.  "Court" means the district court, juvenile'

match = re.match(pattern, string)
if match:
    matched_part = match.group()
    unmatched_part = string[len(matched_part):].lstrip()  # Remove leading spaces
    print("Matched:", matched_part)
    print("Unmatched:", unmatched_part)

In [ ]:
not_matched

In [ ]:
for block in pdf[613].get_text("dict")["blocks"]:
    #print(block['lines'])
    for lines in block['lines']:
        #print(lines['spans'])
        for span in lines['spans']:
            #print(span)
            if span['font']== 'Courier-Bold':
                print(span['text'])

In [ ]:
    # # Create Document with Metadata
    document=[]
    page=st_page
    if edit=="yes":
        length=len(first_remove_statute_text(pdf.load_page(page).get_text()).replace('\n',''))
    else:
        length=len(pdf.load_page(page).get_text().replace('\n',''))
        
    for doc in docs:
        while length<=0:
            page+=1
            if edit=="yes":
                length+=len(" "+remove_statute_text(pdf.load_page(page).get_text()).replace('\n',''))
            else:
                length+=len(" "+pdf.load_page(page).get_text().replace('\n',''))
        document.append(Document(
        page_content=doc,
        metadata={"source": f"{name}", "page": page}
        ))
        length-=len(doc)
        

In [ ]:
import re

def first_remove_statute_text(input_text):
    # Regular expression to match the specific pattern with the fixed date
    pattern = r"\nStatute text rendered on: 1/1/2025\n- \d+ -\n"
    
    # Use re.sub() to replace the matched pattern with an empty string
    cleaned_text = re.sub(pattern, '', input_text)
    
    return cleaned_text


def remove_statute_text(input_text):
    # Regular expression to match the specific pattern with the fixed date
    pattern = r"\nFAMILY CODE\nStatute text rendered on: 1/1/2025\n- \d+ -\n"
    
    # Use re.sub() to replace the matched pattern with an empty string
    cleaned_text = re.sub(pattern, '', input_text)
    
    return cleaned_text


In [ ]:
pdf.load_page(1621).get_text()

In [ ]:
remove_statute_text(pdf.load_page(1621).get_text())

##### Test PDF Pull

In [ ]:
def testing (name,st_page,edit):
    # Load PDF
    # Load PDF
    pdf=fitz.open(f'./pdfs/{name}.pdf')
    
    # Combine Text into one text file       
    text=''
    #for all other
    for i in range(st_page,pdf.page_count):
        if edit=="yes":
            if i==st_page:
                text+=first_remove_statute_text(pdf.load_page(i).get_text())
            else:
                text+=" "+remove_statute_text(pdf.load_page(i).get_text())
        else:
            if i==st_page:
                text+=pdf.load_page(i).get_text()
            else:
                text+=" "+pdf.load_page(i).get_text()
    text=text.replace('\n','')
    print(f"{name}: Text Created")

    # Initialize & Run Semantic Chunker to Break Text into Semantic Chunks
    text_splitter = SemanticChunker(AzureOpenAIEmbeddings(
        openai_api_type="azure",
        openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        deployment="model3large",
        model="text-embedding-3-large",
        chunk_size=1),
        breakpoint_threshold_type="gradient")
    docs=text_splitter.split_text(text)
        
    # # Create Document with Metadata
    document=[]
    page=st_page
    if edit=="yes":
        length=len(first_remove_statute_text(pdf.load_page(page).get_text()).replace('\n',''))
    else:
        length=len(pdf.load_page(page).get_text().replace('\n',''))
        
    for doc in docs:
        print (length)
        while length<=0:
            page+=1
            if edit=="yes":
                length+=len(" "+remove_statute_text(pdf.load_page(page).get_text()).replace('\n',''))
            else:
                length+=len(" "+pdf.load_page(page).get_text().replace('\n',''))
        document.append(Document(
        page_content=doc,
        metadata={"source": f"{name}", "page": page}
        ))
        length-=len(doc)
        
    return text, docs

In [ ]:
text1,docs1=testing(name,st_page,edit)

##### Define Function

In [ ]:
def SemanticEmbeddings (name,st_page,edit):
    # Load PDF
    pdf=fitz.open(f'./pdfs/{name}.pdf')
    
    # Combine Text into one text file       
    text=''
    #for all other
    for i in range(st_page,pdf.page_count):
        if edit=="yes":
            if i==st_page:
                text+=first_remove_statute_text(pdf.load_page(i).get_text())
            else:
                text+=" "+remove_statute_text(pdf.load_page(i).get_text())
        else:
            if i==st_page:
                text+=pdf.load_page(i).get_text()
            else:
                text+=" "+pdf.load_page(i).get_text()
    text=text.replace('\n','')
    print(f"{name}: Text Created")

    # Initialize & Run Semantic Chunker to Break Text into Semantic Chunks
    text_splitter = SemanticChunker(AzureOpenAIEmbeddings(
        openai_api_type="azure",
        openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        deployment="model3large",
        model="text-embedding-3-large",
        chunk_size=1),
        breakpoint_threshold_type="gradient")
    docs=text_splitter.split_text(text)
        
    # # Create Document with Metadata
    document=[]
    page=st_page
    if edit=="yes":
        length=len(first_remove_statute_text(pdf.load_page(page).get_text()).replace('\n',''))
    else:
        length=len(pdf.load_page(page).get_text().replace('\n',''))
        
    for doc in docs:
        while length<=0:
            page+=1
            if edit=="yes":
                length+=len(" "+remove_statute_text(pdf.load_page(page).get_text()).replace('\n',''))
            else:
                length+=len(" "+pdf.load_page(page).get_text().replace('\n',''))
        document.append(Document(
        page_content=doc,
        metadata={"source": f"{name}", "page": page}
        ))
        length-=len(doc)
        
    # Initialize & Run Embeddings Model to Create Embeddings and Store on Harddrive
    embeddings = AzureOpenAIEmbeddings(
            openai_api_type="azure",
            openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
            azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
            deployment="model3large",
            model="text-embedding-3-large",
            chunk_size=1)  
    knowledge_base = Chroma.from_documents(documents=document, persist_directory=f"{name}.pdf",embedding=embeddings)
    print(f"{name}: Embedding Created")  
    
    return knowledge_base

In [ ]:
name="Texas Rules of Civil Procedure"
# Starting Page is page after page index page minus 1
st_page=18
edit="no"

##### Run Semantic Chunking & Create Embeddings

In [ ]:
name="Texas Rules of Civil Procedure"
# Starting Page is page after page index page minus 1
st_page=18
edit="no"

In [ ]:
knowledge_base=SemanticEmbeddings(name,st_page,edit)

In [ ]:
retr=knowledge_base.as_retriever()

In [ ]:
context=retr.invoke("tell me about general provisions")

In [ ]:
print(context[0].page_content)

In [ ]:
context[0].metadata["page"]+1